imports

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../..')))

from ClassificationModel.src.models.classification_model import CancerClassificationModel
from ClassificationModel.src.utils.dataset_utils import load_dataset
from constants.classification.datasets_constants import DatasetConstants
from constants.classification.model_constants import ModelConstants
from utils.callbacks.notification_callback import NotificationCallback
from utils.class_weight_utils import calculate_class_weights
from tensorflow import keras
from ClassificationModel.src.data_processing.image_augmentation import ImageAugmentationPipeline

/Users/ilayn/Ort/FinalsProject/MachineLearning/.venv/lib/python3.13/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):
/Users/ilayn/Ort/FinalsProject/MachineLearning/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading the dataset and creating the model

In [2]:
dataset = load_dataset(base_path=DatasetConstants.DATASETS_DIR / 'unified_dataset_v2')

CHECKPOINT = ModelConstants.CHECKPOINT_FILE_PATH

model = CancerClassificationModel(dataset,
                                  (*DatasetConstants.IMAGE_SIZE, DatasetConstants.CHANNELS)
                                  ,checkpoint_path=CHECKPOINT)

augmenter = ImageAugmentationPipeline()

EPOCHS=50

Found 1192 files belonging to 4 classes.
Found 261 files belonging to 4 classes.
Found 268 files belonging to 4 classes.
Loading checkpoint from: Checkpoints/best_model.keras
✓ Model loaded successfully
  Input shape: (None, 224, 224, 1)
  Output classes: 4


defining callbacks

In [3]:
callbacks = [
    NotificationCallback(
        notifier=model.notifier,
        metrics_to_track=[
            ModelConstants.TRACKED_ACCURACY_METRIC,
            ModelConstants.TRACKED_LOSS_METRIC,
            ModelConstants.TRACKED_VAL_ACCURACY_METRIC,
            ModelConstants.TRACKED_VAL_LOSS_METRIC
        ]),
    keras.callbacks.ModelCheckpoint(
        filepath=ModelConstants.CHECKPOINT_FILE_PATH,
        save_best_only=True,
        monitor=ModelConstants.TRACKED_VAL_LOSS_METRIC
    ),
    keras.callbacks.EarlyStopping(
        monitor=ModelConstants.TRACKED_VAL_LOSS_METRIC,
        patience=15, 
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor=ModelConstants.TRACKED_VAL_LOSS_METRIC,
        factor=0.5,
        patience=5,  
        min_lr=1e-7
    )
]

In [4]:
# Calculate class weights to handle imbalance
class_weights = calculate_class_weights(
    dataset[DatasetConstants.TRAIN_SPLIT_NAME],
    class_names=dataset[DatasetConstants.CLASS_NAMES_KEY]
)

Class weights for handling imbalance:
  adenocarcinoma: 0.774
  large.cell.carcinoma: 1.216
  normal: 1.242
  squamous.cell.carcinoma: 0.925


2026-01-11 10:17:23.766162: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


training the model

In [5]:
history = model.train_model(
    epochs=EPOCHS, 
    callbacks=callbacks, 
    augment_train=True, 
    augmenter=augmenter,
    class_weight=class_weights
)

Epoch 1/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 54s 1s/step - accuracy: 0.9245 - auc: 0.9872 - loss: 0.2347 - precision: 0.9271 - recall: 0.9178 - val_accuracy: 0.8966 - val_auc: 0.9822 - val_loss: 0.3144 - val_precision: 0.8966 - val_recall: 0.8966 - learning_rate: 5.0000e-04
Epoch 2/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 57s 1s/step - accuracy: 0.9690 - auc: 0.9965 - loss: 0.0903 - precision: 0.9706 - recall: 0.9681 - val_accuracy: 0.9387 - val_auc: 0.9961 - val_loss: 0.1461 - val_precision: 0.9387 - val_recall: 0.9387 - learning_rate: 5.0000e-04
Epoch 3/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 58s 2s/step - accuracy: 0.9883 - auc: 0.9997 - loss: 0.0385 - precision: 0.9899 - recall: 0.9857 - val_accuracy: 0.8506 - val_auc: 0.9731 - val_loss: 0.4317 - val_precision: 0.8506 - val_recall: 0.8506 - learning_rate: 5.0000e-04
Epoch 4/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 59s 2s/step - accuracy: 0.9824 - auc: 0.9991 - loss: 0.0496 - precision: 0.9840 - recall: 0.9815 - val_accuracy: 0.9387 - val_auc: 0.9955 - val_loss: 0.1570 -

evaluating the model

In [6]:

results = model.evaluate_model(present_metrics=True,send_message=True,save_confusion_matrix=True)

9/9 ━━━━━━━━━━━━━━━━━━━━ 3s 275ms/step - accuracy: 0.9776 - auc: 0.9998 - loss: 0.0392 - precision: 0.9776 - recall: 0.9776
ACCURACY: 0.978
AUC: 1.000
LOSS: 0.039
PRECISION: 0.978
RECALL: 0.978


2026-01-11 10:53:08.985825: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence



✓ Confusion matrix saved to: results/confusion_matrix_20260111_105309.png

Classification Report:
                         precision    recall  f1-score   support

         adenocarcinoma      0.965     0.965     0.965        85
   large.cell.carcinoma      1.000     1.000     1.000        55
                 normal      1.000     1.000     1.000        57
squamous.cell.carcinoma      0.958     0.958     0.958        71

               accuracy                          0.978       268
              macro avg      0.981     0.981     0.981       268
           weighted avg      0.978     0.978     0.978       268

